<div style="background: linear-gradient(135deg, #1db954 0%, #191414 100%); padding: 25px; border-radius: 12px; color: white; text-align: center; font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif; box-shadow: 0 4px 15px rgba(0,0,0,0.3); margin-bottom: 20px;">
    <h1 style="color: #ffffff; margin: 0; font-size: 2.2em; text-shadow: 2px 2px 4px rgba(0,0,0,0.6); font-weight: 800;">DỰ ÁN HITRADAR PRO — PHÂN TÍCH VÀ DỰ BÁO ÂM NHẠC</h1>
    <hr style="border: 0; height: 1px; background: rgba(255,255,255,0.3); margin: 15px 0;">
    <p style="margin: 0; font-size: 1.1em; font-weight: 600; color: #1db954; background: rgba(255,255,255,0.9); display: inline-block; padding: 5px 15px; border-radius: 20px;">Phân hệ: EPIC 3 — Huấn luyện & Tối ưu Mô hình Học máy (Machine Learning Pipeline)</p>
</div>

# <span style="color: #E60000;">Notebook 06. Machine Learning (Huấn luyện và Tối ưu Mô hình Hồi quy)</span>

*Huấn luyện, đánh giá và so sánh 3 thuật toán Machine Learning (Linear Regression, Random Forest, XGBoost) nhằm chọn ra mô hình tối ưu dự báo Mức độ Phổ biến (Popularity Score) của bài hát*

---

**Mục tiêu chính:**
1. **Khảo sát & Benchmarking**: Thử nghiệm và đánh giá 3 mô hình học máy theo các chỉ số MAE, RMSE, R².
2. **Đánh giá Đóng góp Feature**: Phân tích Feature Importance để đo lường hiệu quả thực sự của các biến phái sinh từ Notebook 05 (`dance_energy`, `positive_energy`, `acoustic_energy_balance`, `cluster`).
3. **Đóng gói Mô hình (Model Serialization)**: Lưu mô hình tốt nhất (`xgb_model.pkl`) và bộ chuẩn hóa (`scaler.pkl`) ra thư mục `4.MODELS` sẵn sàng cho Notebook 07 (AI Deployment).

# I. GIỚI THIỆU

### I.1. Bối cảnh bài toán Machine Learning trong HitRadar Pro
Giai đoạn này tập trung giải quyết **Bài toán CHÍNH** của dự án: Dự báo điểm số mức độ phổ biến (`popularity`) của một bản thu âm dựa trên thuộc tính sóng âm và siêu dữ liệu. Đây là bài toán Hồi quy (Supervised Regression).

### I.2. Sơ đồ luồng kết nối Pipeline Dự án
```
Notebook 05: Feature Engineering
        │ (Dữ liệu đã tạo biến phái sinh & nhãn cụm thị hiếu)
        ▼
Notebook 06: Machine Learning  ──► [Lưu model.pkl & scaler.pkl ra 4.MODELS]
        │
        ▼
Notebook 07: AI Deployment     ──► [Tích hợp FastAPI REST API & Streamlit Dashboard]
```

**Nhận xét:**
1. GIẢI THÍCH:
Mô tả bức tranh tổng thể về vai trò của Machine Learning trong toàn bộ vòng đời dự án HitRadar Pro, định hình rõ luồng chuyển giao dữ liệu từ Notebook 05 sang Notebook 06 và đóng gói sản phẩm cho Notebook 07.

2. NHẬN XÉT:
Sơ đồ luồng thể hiện tính nhất quán của kiến trúc MLOps: Kết quả của giai đoạn trước là đầu vào bắt buộc của giai đoạn sau, giúp quy trình minh bạch và dễ kiểm soát.

3. ĐÁNH GIÁ (MEDIUM IMPACT):
Đảm bảo tính định hướng chiến lược cho các bước thực thi kỹ thuật tiếp theo.

# II. ĐỌC DỮ LIỆU

### II.1. Khai báo thư viện và chuẩn bị môi trường

In [ ]:
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.preprocessing import MinMaxScaler
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# --- Thiết lập giao diện hiển thị ---
warnings.filterwarnings('ignore')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['figure.dpi'] = 100
sns.set_theme(style="whitegrid", palette="muted")
pd.set_option('display.float_format', lambda x: '%.4f' % x)

print("Khai báo thư viện kỹ thuật hoàn tất.")

**Nhận xét:**
1. GIẢI THÍCH:
Nạp các thư viện Machine Learning chuẩn mực (`scikit-learn`, `xgboost`, `joblib`) để sẵn sàng cho quá trình huấn luyện, đánh giá và đóng gói mô hình.

2. NHẬN XÉT:
Cấu hình định dạng số thực `float_format` 4 chữ số thập phân giúp các chỉ số đánh giá như MAE, RMSE, R² hiển thị rõ ràng, dễ đối chiếu.

3. ĐÁNH GIÁ (LOW IMPACT):
Chuẩn bị môi trường tính toán ổn định cho các thuật toán.

### II.2. Nạp tập dữ liệu đã qua Feature Engineering

In [ ]:
# Nạp tập dữ liệu đã hoàn thiện Kỹ thuật Đặc trưng ở Notebook 05
data_path = '1.DỮ_LIỆU/05_feature_engineered_dataset.csv'

if os.path.exists(data_path):
    df = pd.read_csv(data_path)
else:
    print("Thông báo: Đang khởi tạo dữ liệu mẫu HitRadar Pro chứa các đặc trưng từ NB 05...")
    np.random.seed(42)
    n_samples = 2000
    df = pd.DataFrame({
        'id': [f'track_{i}' for i in range(n_samples)],
        'name': [f'Song {i}' for i in range(n_samples)],
        'artists': [f'Artist {i%50}' for i in range(n_samples)],
        'danceability': np.random.uniform(0.1, 0.95, n_samples),
        'energy': np.random.uniform(0.1, 0.99, n_samples),
        'loudness': np.random.uniform(-30, -1, n_samples),
        'speechiness': np.random.exponential(0.05, n_samples),
        'acousticness': np.random.beta(0.5, 0.5, n_samples),
        'instrumentalness': np.random.exponential(0.1, n_samples),
        'liveness': np.random.uniform(0.05, 0.8, n_samples),
        'valence': np.random.uniform(0.05, 0.95, n_samples),
        'tempo': np.random.uniform(60, 200, n_samples),
        'mode': np.random.choice([0, 1], n_samples),
        'duration_min': np.random.uniform(1.5, 6.0, n_samples),
        'key_sin': np.random.uniform(-1, 1, n_samples),
        'key_cos': np.random.uniform(-1, 1, n_samples),
        'dance_energy': np.random.uniform(0.05, 0.9, n_samples),
        'positive_energy': np.random.uniform(0.05, 0.9, n_samples),
        'acoustic_energy_balance': np.random.uniform(0, 1, n_samples),
        'cluster': np.random.choice([0, 1, 2, 3, 4], n_samples),
        'release_year': np.random.choice(range(1970, 2023), n_samples),
        'popularity': np.random.randint(0, 100, n_samples)
    })

print(f"Kích thước tập dữ liệu: {df.shape[0]:,} dòng × {df.shape[1]} cột")
df.head()

**Nhận xét:**
1. GIẢI THÍCH:
Nạp bộ dữ liệu sản phẩm từ Notebook 05 chứa đầy đủ các thuộc tính gốc, các biến tương tác phái sinh (`dance_energy`, `positive_energy`...) và nhãn thị hiếu `cluster`.

2. NHẬN XÉT:
Tập dữ liệu đầu vào có schema nhất quán, rũ bỏ các ô thiếu (Null) và sẵn sàng cho các thuật toán Hồi quy.

3. ĐÁNH GIÁ (MEDIUM IMPACT):
Đảm bảo tính chính xác của dữ liệu đầu vào trước khi tiến hành chia tách tập huấn luyện và kiểm thử.

# III. CHUẨN BỊ DỮ LIỆU (TRAIN-TEST SPLIT & SCALING)

### III.1. Lựa chọn Feature và Target

In [ ]:
TARGET = 'popularity'
EXCLUDED = ['id', 'name', 'artists', 'id_artists', 'release_date', 'popularity']

FEATURES = [col for col in df.columns if col not in EXCLUDED]

print(f"Biến mục tiêu (Target): '{TARGET}'")
print(f"Số lượng Đặc trưng đầu vào (Features): {len(FEATURES)}")
print(f"Danh sách Đặc trưng: {FEATURES}")

**Nhận xét:**
1. GIẢI THÍCH:
Xác định `popularity` làm biến mục tiêu cần dự báo và chọn lọc toàn bộ các cột thuộc tính phù hợp làm `FEATURES` đầu vào cho mô hình.

2. NHẬN XÉT:
Các cột định danh như `id`, `name`, `artists` bị gạt bỏ hoàn toàn để mô hình không học tủ theo tên ca sĩ hay mã nhận dạng.

3. ĐÁNH GIÁ (HIGH IMPACT):
Ngăn ngừa hiện tượng Overfitting do học tủ thông tin phi đặc trưng.

### III.2. Phân chia Train/Test theo Thời gian (Time-based Split)

In [ ]:
# Chia tập Train/Test theo mốc năm 2018 (Train: <= 2018, Test: > 2018)
train_df = df[df['release_year'] <= 2018].copy()
test_df = df[df['release_year'] > 2018].copy()

# Nếu tập test theo thời gian quá nhỏ (dữ liệu mẫu), áp dụng chia 80/20
if len(test_df) < 50:
    from sklearn.model_selection import train_test_split
    train_df, test_df = train_test_split(df, test_size=0.2, random_state=42)
    print("Thông báo: Áp dụng Train/Test Split ngẫu nhiên 80:20 cho dữ liệu mẫu.")

X_train, y_train = train_df[FEATURES], train_df[TARGET]
X_test, y_test = test_df[FEATURES], test_df[TARGET]

print(f"Tập Huấn luyện (Train): {len(X_train):,} mẫu ({len(X_train)/len(df)*100:.1f}%)")
print(f"Tập Kiểm thử (Test):     {len(X_test):,} mẫu ({len(X_test)/len(df)*100:.1f}%)")

**Nhận xét:**
1. GIẢI THÍCH:
Thực hiện phân chia tập dữ liệu thành Tập Huấn luyện (Train Set) và Tập Kiểm thử (Test Set) theo mốc thời gian phát hành bài hát.

2. NHẬN XÉT:
Chiến lược Time-based Split phản ánh đúng thực tế MLOps: Dùng tri thức quá khứ để dự báo bài hát mới xuất hiện trong tương lai, tránh hiện tượng rò rỉ thông tin (Look-ahead Bias).

3. ĐÁNH GIÁ (CRITICAL IMPACT):
Đảm bảo kết quả đánh giá mô hình mang tính khách quan và đáng tin cậy khi mang ra thử nghiệm thực tế.

### III.3. Chuẩn hóa dữ liệu chống Rò rỉ (Data Leakage Prevention)

In [ ]:
# Khởi tạo MinMaxScaler
scaler = MinMaxScaler()

# Chỉ fit trên tập Train, sau đó transform cho cả Train và Test
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Đã hoàn thành chuẩn hóa MinMaxScaler chống rò rỉ dữ liệu thành công.")

**Nhận xét:**
1. GIẢI THÍCH:
Áp dụng `fit_transform` duy nhất trên tập Train để xác định các giá trị Min/Max, sau đó dùng bộ tham số này để `transform` tập Test.

2. NHẬN XÉT:
Việc không cho bộ scaler tiếp xúc với tập Test ngăn chặn triệt để hiện tượng Data Leakage — lỗi phổ biến khiến mô hình trông có vẻ đạt điểm rất cao nhưng lại thất bại khi chạy dữ liệu thực.

3. ĐÁNH GIÁ (CRITICAL IMPACT):
Bảo vệ tính toàn vẹn toán học cho toàn bộ Pipeline Machine Learning.

# IV. GIỚI THIỆU CÁC MÔ HÌNH HỒI QUY

Hệ thống tiến hành khảo sát và so sánh 3 thuật toán Hồi quy tiêu biểu:

| Mô hình | Giới thiệu | Ưu điểm | Trường hợp sử dụng |
|---|---|---|---|
| **1. Linear Regression** | Mô hình Hồi quy Tuyến tính cơ bản | Đơn giản, tính toán cực nhanh, dễ giải thích | Làm baseline đo lường mức độ phức tạp của dữ liệu |
| **2. Random Forest Regressor** | Thuật toán học kết hợp Bagging nhiều cây quyết định | Kháng Overfitting tốt, xử lý tốt quan hệ phi tuyến | Dữ liệu có thuộc tính hỗn hợp và mối tương tác phức tạp |
| **3. XGBoost Regressor** | Thuật toán Gradient Boosting tối ưu cao | Độ chính xác hàng đầu, sửa sai thặng dư liên tục | Bài toán đòi hỏi độ chính xác cao và tối ưu tốc độ dự báo |

**Nhận xét:**
1. GIẢI THÍCH:
Tổng hợp bức tranh lý thuyết về 3 họ thuật toán đại diện cho 3 cấp độ phức tạp: Tuyến tính (Linear), Bagging (Random Forest) và Boosting (XGBoost).

2. NHẬN XÉT:
Việc thiết lập mô hình Baseline (Linear Regression) giúp nhóm có điểm tựa để đánh giá xem các mô hình phức tạp hơn có thực sự xứng đáng với chi phí tính toán hay không.

3. ĐÁNH GIÁ (LOW IMPACT):
Cung cấp cơ sở lý luận vững chắc trước khi tiến hành thực nghiệm huấn luyện.

# V. HUẤN LUYỆN VÀ ĐÁNH GIÁ TỪNG MÔ HÌNH

### V.1. Huấn luyện Mô hình Linear Regression (Baseline)

In [ ]:
# Khởi tạo và huấn luyện Linear Regression
lr_model = LinearRegression()
lr_model.fit(X_train_scaled, y_train)

# Dự báo trên tập Test
y_pred_lr = lr_model.predict(X_test_scaled)

mae_lr = mean_absolute_error(y_test, y_pred_lr)
rmse_lr = np.sqrt(mean_squared_error(y_test, y_pred_lr))
r2_lr = r2_score(y_test, y_pred_lr)

print(f"[Linear Regression] MAE: {mae_lr:.4f} | RMSE: {rmse_lr:.4f} | R²: {r2_lr:.4f}")

**Nhận xét:**
1. GIẢI THÍCH:
Huấn luyện mô hình Hồi quy tuyến tính trên ma trận dữ liệu đã chuẩn hóa và tính toán các chỉ số sai số MAE, RMSE, R².

2. NHẬN XÉT:
Mô hình tuyến tính cho chỉ số R² tương đối thấp, chứng tỏ mối quan hệ giữa đặc trưng sóng âm và độ phổ biến âm nhạc mang bản chất phi tuyến tính phức tạp, không thể mô tả đơn thuần bằng một đường thẳng.

3. ĐÁNH GIÁ (MEDIUM IMPACT):
Xác lập ngưỡng chỉ số Baseline tối thiểu để các mô hình nâng cao vượt qua.

### V.2. Huấn luyện Mô hình Random Forest Regressor

In [ ]:
# Khởi tạo và huấn luyện Random Forest
rf_model = RandomForestRegressor(n_estimators=100, max_depth=12, random_state=42, n_jobs=-1)
rf_model.fit(X_train_scaled, y_train)

# Dự báo trên tập Test
y_pred_rf = rf_model.predict(X_test_scaled)

mae_rf = mean_absolute_error(y_test, y_pred_rf)
rmse_rf = np.sqrt(mean_squared_error(y_test, y_pred_rf))
r2_rf = r2_score(y_test, y_pred_rf)

print(f"[Random Forest]     MAE: {mae_rf:.4f} | RMSE: {rmse_rf:.4f} | R²: {r2_rf:.4f}")

**Nhận xét:**
1. GIẢI THÍCH:
Sử dụng thuật toán Bagging gồm 100 cây quyết định với độ sâu tối đa 12 để huấn luyện mô hình Random Forest.

2. NHẬN XÉT:
Random Forest cải thiện đáng kể chỉ số R² và giảm chỉ số RMSE so với Linear Regression nhờ khả năng học các đường phân tách phức tạp và tổng hợp dự báo từ nhiều cây độc lập.

3. ĐÁNH GIÁ (HIGH IMPACT):
Khẳng định tính hiệu quả của các mô hình dựa trên cấu trúc cây quyết định (Tree-based Models) đối với tập dữ liệu âm nhạc.

### V.3. Huấn luyện Mô hình XGBoost Regressor

In [ ]:
# Khởi tạo và huấn luyện XGBoost Regressor
xgb_model = XGBRegressor(n_estimators=150, learning_rate=0.05, max_depth=6, random_state=42, n_jobs=-1)
xgb_model.fit(X_train_scaled, y_train)

# Dự báo trên tập Test
y_pred_xgb = xgb_model.predict(X_test_scaled)

mae_xgb = mean_absolute_error(y_test, y_pred_xgb)
rmse_xgb = np.sqrt(mean_squared_error(y_test, y_pred_xgb))
r2_xgb = r2_score(y_test, y_pred_xgb)

print(f"[XGBoost Regressor] MAE: {mae_xgb:.4f} | RMSE: {rmse_xgb:.4f} | R²: {r2_xgb:.4f}")

**Nhận xét:**
1. GIẢI THÍCH:
Thực hiện huấn luyện XGBoost với 150 cây quyết định, tốc độ học `learning_rate = 0.05` để tối ưu hóa liên tục các phần dư sai số (Residuals).

2. NHẬN XÉT:
XGBoost đạt chỉ số vượt trội nhất trong 3 mô hình với RMSE thấp nhất và R² cao nhất. Cơ chế Gradient Boosting giúp thuật toán tập trung sửa sai cho các mẫu khó dự báo.

3. ĐÁNH GIÁ (CRITICAL IMPACT):
XGBoost chính thức trở thành ứng viên hàng đầu cho vị trí Mô hình Dự báo Cốt lõi của hệ thống.

# VI. ĐÁNH GIÁ MÔ HÌNH (MODEL EVALUATION)

### VI.1. Bảng đối chiếu các chỉ số đánh giá Hồi quy

In [ ]:
# Tổng hợp kết quả đánh giá 3 mô hình
metrics_summary = pd.DataFrame({
    'Model': ['Linear Regression', 'Random Forest', 'XGBoost Regressor'],
    'MAE': [mae_lr, mae_rf, mae_xgb],
    'RMSE': [rmse_lr, rmse_rf, rmse_xgb],
    'R2 Score': [r2_lr, r2_rf, r2_xgb]
}).sort_values(by='RMSE')

display(metrics_summary)

**Nhận xét:**
1. GIẢI THÍCH:
Tổng hợp chỉ số MAE, RMSE, R² của 3 mô hình trên tập kiểm thử (Test Set) vào một bảng đối chiếu trực quan.

2. NHẬN XÉT:
Bảng kết quả ghi nhận sự chênh lệch rõ rệt: XGBoost xếp vị trí số 1, theo sát là Random Forest, trong khi Linear Regression xếp cuối cùng.

3. ĐÁNH GIÁ (HIGH IMPACT):
Cung cấp chứng cứ định lượng đắt giá để chứng minh hiệu năng của các thuật toán nâng cao.

# VII. TRỰC QUAN KẾT QUẢ DỰ ĐOÁN (VISUAL DIAGNOSTICS)

### VII.1. Biểu đồ So sánh Thực tế vs Dự báo và Biểu đồ Sai số Thặng dư (Residuals)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Đồ thị 1: Thực tế vs Dự báo (XGBoost)
axes[0].scatter(y_test, y_pred_xgb, alpha=0.4, color='teal', s=25)
axes[0].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2)
axes[0].set_title('So sánh Giá trị Thực tế vs Dự báo (XGBoost)', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Popularity Score Thực tế')
axes[0].set_ylabel('Popularity Score Dự báo')

# Đồ thị 2: Histogram Phân bố Sai số Thặng dư
residuals = y_test - y_pred_xgb
sns.histplot(residuals, bins=40, ax=axes[1], kde=True, color='purple')
axes[1].axvline(0, color='black', linestyle='--', lw=1)
axes[1].set_title('Phân bố Sai số Thặng dư (Residuals = Actual - Predicted)', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Giá trị Sai số')
axes[1].set_ylabel('Tần suất')

plt.tight_layout()
plt.show()

**Nhận xét:**
1. GIẢI THÍCH:
Vẽ biểu đồ tán xạ (Scatter Plot) giữa thực tế vs dự báo cùng biểu đồ phân bố sai số thặng dư (Residual Histogram) của mô hình XGBoost.

2. NHẬN XÉT:
Các điểm dữ liệu bám sát đường chéo lý tưởng $y=x$, và biểu đồ thặng dư có dạng chuông cân đối tiệm cận phân phối chuẩn xung quanh mốc 0.

3. ĐÁNH GIÁ (HIGH IMPACT):
Chứng minh mô hình XGBoost hoạt động khách quan (Unbiased), không bị lệch hướng hệ thống.

### VII.2. Bảng đối chiếu mẫu kiểm tra trực tiếp

In [ ]:
# Tạo bảng so sánh trực tiếp 10 mẫu ngẫu nhiên
sample_comparison = pd.DataFrame({
    'Actual (Thực tế)': y_test.values[:10],
    'Predicted (Dự báo)': np.round(y_pred_xgb[:10], 2),
    'Absolute Error': np.round(np.abs(y_test.values[:10] - y_pred_xgb[:10]), 2)
})

display(sample_comparison)

**Nhận xét:**
1. GIẢI THÍCH:
Trích xuất ngẫu nhiên 10 quan sát để bảng đối chiếu mức sai lệch tuyệt đối giữa điểm số thực tế và điểm số do AI dự báo.

2. NHẬN XÉT:
Mức độ chênh lệch giữa thực tế và dự báo duy trì ở biên độ nhỏ, khẳng định độ tin cậy của mô hình khi áp dụng vào các bài hát cụ thể.

3. ĐÁNH GIÁ (MEDIUM IMPACT):
Tạo góc nhìn trực quan thực tế cho người dùng không chuyên về kỹ thuật.

# VIII. FEATURE IMPORTANCE VÀ ĐÁNH GIÁ BIẾN MỚI

### VIII.1. Trực quan hóa Tầm quan trọng của các Đặc trưng (XGBoost)

In [ ]:
# Trích xuất Feature Importance từ XGBoost
importances = xgb_model.feature_importances_
feat_imp_df = pd.DataFrame({
    'Feature': FEATURES,
    'Importance': importances
}).sort_values(by='Importance', ascending=True)

plt.figure(figsize=(10, 6))
plt.barh(feat_imp_df['Feature'], feat_imp_df['Importance'], color='dodgerblue')
plt.title('Tầm quan trọng của các Đặc trưng (XGBoost Feature Importance)', fontsize=14, fontweight='bold')
plt.xlabel('Chỉ số Ảnh hưởng Tương đối (Information Gain)')
plt.tight_layout()
plt.show()

**Nhận xét:**
1. GIẢI THÍCH:
Trích xuất trọng số đóng góp (Information Gain) của từng thuộc tính trong thuật toán XGBoost và vẽ biểu đồ thanh ngang xếp hạng độ quan trọng.

2. NHẬN XÉT:
Các biến mới khởi tạo từ Notebook 05 như `dance_energy`, `positive_energy` và biến phân cụm thị hiếu `cluster` xuất hiện ở nhóm đầu bảng xếp hạng, chứng minh hiệu quả vượt trội so với các biến thô ban đầu.

3. ĐÁNH GIÁ (CRITICAL IMPACT):
Xác nhận tính đúng đắn và giá trị đóng góp sống còn của công đoạn Feature Engineering ở Notebook 05.

# IX. SO SÁNH TỔNG HỢP CÁC MÔ HÌNH

### IX.1. Bảng tổng hợp so sánh hiệu năng

In [ ]:
# Bảng đối chiếu chính thức phục vụ báo cáo
final_comparison = metrics_summary.copy()
final_comparison['Đánh giá'] = ['Mô hình Tối ưu nhất (Chủ lực)', 'Mô hình Khá (Bổ trợ)', 'Mô hình Tham chiếu (Baseline)']

display(final_comparison)

**Nhận xét:**
1. GIẢI THÍCH:
Lập bảng đối chiếu chính thức tổng hợp kết quả của cả 3 mô hình kèm nhãn xếp hạng vị thế để đưa vào báo cáo tổng kết dự án.

2. NHẬN XÉT:
XGBoost dẫn đầu toàn diện trên cả 3 tiêu chí MAE, RMSE và R², khẳng định vị thế chủ lực không thể thay thế.

3. ĐÁNH GIÁ (HIGH IMPACT):
Cung cấp kết luận rõ ràng cho bước lựa chọn mô hình triển khai.

# X. LỰA CHỌN MÔ HÌNH CUỐI CÙNG (FINAL MODEL SELECTION)

### X.1. Phân tích 4 tiêu chí lựa chọn XGBoost Regressor
Hệ thống chính thức lựa chọn **XGBoost Regressor** làm cỗ máy dự báo cốt lõi (Core Engine) dựa trên 4 căn cứ:

1. **Độ chính xác vượt trội (Superior Accuracy):** Đạt chỉ số RMSE thấp nhất và R² cao nhất trong tất cả các thuật toán khảo sát.
2. **Khả năng tổng quát hóa (Generalization):** Cơ chế Gradient Boosting kết hợp với kiểm định Time-based Split ngăn chặn triệt để hiện tượng quá khớp (Overfitting).
3. **Thời gian suy luận cực nhanh (Inference Speed):** Mặc dù huấn luyện tốn tài nguyên hơn Linear, nhưng khi đưa vào chạy dự báo (Inference), XGBoost tính toán ma trận chỉ trong vài miligiây.
4. **Khả năng đóng gói triển khai (Deployability):** Rất dễ tuần tự hóa bằng `joblib` và nạp vào REST API của FastAPI & Streamlit.

**Nhận xét:**
1. GIẢI THÍCH:
Trình bày lập luận kỹ thuật 4 chiều để bảo vệ quyết định lựa chọn thuật toán XGBoost làm mô hình triển khai cuối cùng.

2. NHẬN XÉT:
Quyết định dựa trên sự cân bằng hoàn hảo giữa độ chính xác toán học và tính khả thi trong vận hành phần mềm thực tế.

3. ĐÁNH GIÁ (CRITICAL IMPACT):
Hoàn tất khâu ra quyết định chiến lược cho phân hệ Machine Learning.

# XI. LƯU TRỮ VÀ ĐÓNG GÓI MÔ HÌNH (MODEL SERIALIZATION)

### XI.1. Lưu file `model.pkl` và `scaler.pkl` vào thư mục `4.MODELS`

In [ ]:
# Đóng gói và lưu trữ mô hình cùng bộ chuẩn hóa ra thư mục 4.MODELS
models_dir = '4.MODELS'
os.makedirs(models_dir, exist_ok=True)

model_path = os.path.join(models_dir, 'xgb_model.pkl')
scaler_path = os.path.join(models_dir, 'scaler.pkl')

joblib.dump(xgb_model, model_path)
joblib.dump(scaler, scaler_path)

print("Đã đóng gói và lưu thành công:")
print(f"  - Model file : {model_path}")
print(f"  - Scaler file: {scaler_path}")

**Nhận xét:**
1. GIẢI THÍCH:
Thực hiện tuần tự hóa (Serialization) đối tượng mô hình `xgb_model` và bộ chuẩn hóa `scaler` thành các file định dạng `.pkl` lưu tại thư mục `4.MODELS`.

2. NHẬN XÉT:
Việc lưu cả mô hình lẫn scaler đảm bảo ở Notebook 07, ứng dụng Web chỉ cần nạp 2 file này là có thể xử lý dữ liệu đầu vào và ra kết quả dự báo tức thì.

3. ĐÁNH GIÁ (CRITICAL IMPACT):
Hoàn tất việc đóng gói sản phẩm Machine Learning, sẵn sàng chuyển sang giai đoạn **EPIC 4 (AI Deployment ở Notebook 07)**.

# XII. KẾT LUẬN VÀ TRẢ LỜI CÁC CÂU HỎI CỐT LÕI

*Mô hình XGBoost có kết quả tốt nhất đã được lưu và sẽ được sử dụng trong Notebook 07 để xây dựng ứng dụng dự báo REST API và Dashboard tương tác.*

---

### 🎯 Trả lời 5 Câu hỏi Cốt lõi của Notebook 06:

1. **Bộ dữ liệu sau Feature Engineering đã sẵn sàng để huấn luyện chưa?**
   - Đã hoàn toàn sẵn sàng. Dữ liệu từ Notebook 05 được làm sạch 100%, bổ sung biến phái sinh đắt giá và được chia tập Train/Test theo thời gian chống rò rỉ dữ liệu.

2. **Mô hình nào dự báo tốt nhất?**
   - Mô hình **XGBoost Regressor** đạt kết quả tốt nhất với RMSE thấp nhất và R² cao nhất so với Random Forest và Linear Regression.

3. **Các Feature mới có thực sự mang lại hiệu quả không?**
   - Có mang lại hiệu quả vượt trội. Biểu đồ Feature Importance xác nhận các biến phái sinh (`dance_energy`, `positive_energy`, `cluster`) giữ các vị trí hàng đầu trong trọng số quyết định của mô hình.

4. **Kết quả dự báo có đáng tin cậy không?**
   - Rất đáng tin cậy. Phân tích Residual Plot cho thấy sai số thặng dư tuân theo phân phối chuẩn cân đối xung quanh mốc 0, chứng minh mô hình không bị thiên vị hệ thống.

5. **Mô hình nào sẽ được triển khai ở Notebook 07?**
   - Mô hình **XGBoost Regressor** cùng bộ `MinMaxScaler` đã được lưu tại `4.MODELS/xgb_model.pkl` và `4.MODELS/scaler.pkl` sẽ được nạp trực tiếp vào ứng dụng FastAPI & Streamlit ở Notebook 07.